[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/cadquery_basics/blob/master/docs/tuto_colab_build/2d_to_3d.ipynb)

# 3D a partir de 2D
## build123d para Arquitetos e Engenheiros
### Versão Google Colab

Este notebook apresenta exemplos de construção de sólidos utilizando polígonos, curvas e as operações `extrude`, `sweep` e `loft` da biblioteca build123d. A visualização é feita com o `build123d_simpleviewer`.

## Instalação

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Colab, installing packages...")
    !pip install build123d
    !wget -q -N https://raw.githubusercontent.com/255ribeiro/cadquery_basics/master/docs/tuto_colab_build/build123d_simpleviewer.py
else:
    print("Not running in Colab, skipping package installation.")

## Importação dos pacotes

In [ ]:
# Instalação (se necessário - descomente caso os pacotes não estejam disponíveis)
# !pip install build123d

from build123d import *
from build123d_simpleviewer import show

## 1. Extrude – Extrusão de um polígono

Criamos um hexágono regular e extrudamos para formar um prisma hexagonal.

> ⚠️ **Atenção ao parâmetro do polígono**: o `.polygon(nSides, diameter)` do CadQuery recebe o **diâmetro** circunscrito. O `RegularPolygon(radius, side_count)` do build123d recebe o **raio** — metade do diâmetro. Por isso `polygon(6, 10)` (diâmetro 10) vira `RegularPolygon(radius=5, side_count=6)` (raio 5).

In [ ]:
# Definir o polígono (hexágono) no plano XY
hexagon = RegularPolygon(radius=5, side_count=6)  # 6 lados, raio 5 (diâmetro 10)

# Extrudar para 15 mm de altura
extruded_hex = extrude(hexagon, amount=15)

# Exibir o sólido
show(extruded_hex)

## 2. Sweep – Varredura de um perfil ao longo de uma curva

Criamos um perfil circular e o varremos ao longo de uma curva (spline) definida por pontos. No build123d o caminho é desenhado dentro de um `BuildLine`, e o resultado (`.line`) é passado para `sweep()`.

In [ ]:
# Perfil: círculo de raio 2
profile = Circle(2)

# Caminho: curva spline passando pelos pontos (0,0,0), (5,5,5), (10,0,10)
with BuildLine() as path_line:
    Spline((0, 0, 0), (5, 5, 5), (10, 0, 10))

# Varredura do perfil ao longo do caminho
swept = sweep(profile, path=path_line.line, is_frenet=True)

# Exibir o resultado
show(swept)

## 3. Loft – Loft entre dois polígonos

Criamos um quadrado na base e um triângulo no topo, e realizamos um loft entre eles. `loft()` aceita diretamente uma lista de `Sketch`/`Face` — sem precisar de `.add()` / `toPending()`.

In [ ]:
# Perfil inferior: quadrado de lado 10
bottom = Rectangle(10, 10)

# Perfil superior: triângulo equilátero (polígono de 3 lados) deslocado em Z
top = Plane(origin=(0, 0, 15)) * RegularPolygon(radius=6, side_count=3)  # raio 6, centro no mesmo XY

# Loft entre os dois perfis
lofted = loft([bottom, top])

# Exibir
show([top, bottom, lofted],
     tessellation_tolerance=.001)

## 4. Combinação – Extrusão de polígono com furo e sweep com perfil poligonal

Exemplo mais avançado: criamos uma base extrusada com um furo e depois fazemos um sweep com um perfil poligonal (pentágono) ao longo de um arco.

O `.faces(">Z")` do CadQuery vira `.faces().sort_by(Axis.Z)[-1]` no build123d — mais verboso, porém mais explícito sobre qual face está sendo selecionada. E como já sabemos a espessura da base, furamos com uma extrusão negativa da mesma profundidade em vez do `.cutThruAll()`.

In [ ]:
# Base: um retângulo extrudado com furo circular
base = extrude(Rectangle(30, 20), amount=5)

face_topo = base.faces().sort_by(Axis.Z)[-1]
furo      = Plane(face_topo) * Circle(5)
base      = base - extrude(furo, amount=-5)   # perfura de cima para baixo, atravessando os 5mm de espessura

# Caminho para sweep: arco de 90 graus no plano XZ, deslocado para (0, 5, 0)
with BuildLine(Plane(origin=(0, 5, 0), x_dir=(1, 0, 0), z_dir=(0, -1, 0))) as path_arc_line:
    RadiusArc((0, 0), (10, 10), 90)

# Perfil: pentágono regular (diâmetro 2.5 → raio 1.25)
poly_profile = RegularPolygon(radius=1.25, side_count=5)

# Sweep do pentágono ao longo do arco
swept_poly = sweep(poly_profile, path=path_arc_line.line, is_frenet=True)

# Combinar a base com o sweep (unir os sólidos)
result = base + swept_poly

# Exibir o resultado final
show(result)

## 5. Loft com múltiplas seções e curvas

Demonstração de loft entre três polígonos diferentes: um quadrado, um octógono e um círculo, com alturas variadas.

In [ ]:
# Seção inferior: quadrado de lado 8
s1 = Rectangle(8, 8)

# Seção intermediária: octógono regular, raio 6, na altura 10
s2 = Plane(origin=(0, 0, 10)) * RegularPolygon(radius=6, side_count=8)

# Seção superior: círculo de raio 5, na altura 20
s3 = Plane(origin=(0, 0, 20)) * Circle(5)

# Loft entre as três seções
multi_loft = loft([s1, s2, s3])

# Exibir
show(multi_loft)